In [1]:
import os
import json
import numpy as np
import pandas as pd


In [25]:
def list_directories(path):
    items = os.listdir(path)
    directories = [item for item in items if os.path.isdir(os.path.join(path, item))]
    
    return directories

path = 'data'
directories = list_directories(path)

In [26]:
def load_json_files(data, dir_path):
    for filename in os.listdir(dir_path):
        if filename.endswith('.json'):
            filepath = os.path.join(dir_path, filename)
            with open(filepath, 'r') as file:
                json_data = json.load(file)
                data.append(json_data)
    return data


In [27]:
def process_data(data, label):
    result = []
    for datum in data:
        datum_dict = {}
        datum_dict['label'] = label
        datum_dict['sequences'] = []      
        for sequence in datum:
            sequence_dict = {}
            sequence_dict['seq'] = int(sequence['seq'])
            sequence_dict['landmarks'] = []
            for landmark in sequence['hand_landmarks']:
                sequence_dict['landmarks'].append({
                    'id': landmark['id'],
                    'x': round(landmark['x'], 3),
                    'y': round(landmark['y'], 3),
                    'z': round(landmark.get('z', 0), 3)
                })
            sequence_dict['hand_label'] = sequence['hand_label']
            sequence_dict['label'] = label
            datum_dict['sequences'].append(sequence_dict)
        result.append(datum_dict)
    return result


In [28]:
dataset = []
for directory in directories:
    dir_path = os.path.join(path, directory)
    data = []
    data = load_json_files(data, dir_path)
    label = directory
    processed_data = process_data(data, label)
    dataset.extend(processed_data)



In [29]:
import pprint
print(dataset[10])

{'label': 'rotate-right', 'sequences': [{'seq': 0, 'landmarks': [{'id': 0, 'x': 0.66, 'y': 0.811, 'z': -0.0}, {'id': 1, 'x': 0.705, 'y': 0.726, 'z': 0.023}, {'id': 2, 'x': 0.707, 'y': 0.646, 'z': 0.028}, {'id': 3, 'x': 0.686, 'y': 0.581, 'z': 0.031}, {'id': 4, 'x': 0.656, 'y': 0.555, 'z': 0.033}, {'id': 5, 'x': 0.689, 'y': 0.596, 'z': -0.005}, {'id': 6, 'x': 0.662, 'y': 0.518, 'z': -0.0}, {'id': 7, 'x': 0.661, 'y': 0.546, 'z': 0.007}, {'id': 8, 'x': 0.667, 'y': 0.576, 'z': 0.012}, {'id': 9, 'x': 0.641, 'y': 0.611, 'z': -0.016}, {'id': 10, 'x': 0.622, 'y': 0.535, 'z': -0.004}, {'id': 11, 'x': 0.625, 'y': 0.561, 'z': 0.009}, {'id': 12, 'x': 0.632, 'y': 0.59, 'z': 0.014}, {'id': 13, 'x': 0.597, 'y': 0.637, 'z': -0.022}, {'id': 14, 'x': 0.585, 'y': 0.567, 'z': -0.008}, {'id': 15, 'x': 0.594, 'y': 0.591, 'z': 0.008}, {'id': 16, 'x': 0.601, 'y': 0.618, 'z': 0.014}, {'id': 17, 'x': 0.561, 'y': 0.67, 'z': -0.027}, {'id': 18, 'x': 0.556, 'y': 0.604, 'z': -0.012}, {'id': 19, 'x': 0.566, 'y': 0.6

In [31]:
import numpy as np
import random
from scipy.interpolate import interp1d

hand_data = dataset
def parse_hand_data(data):
    sequences = []
    for sample in data:
        for seq in sample['sequences']:
            landmarks = seq['landmarks']
            sequence = np.array([[lm['x'], lm['y'], lm['z']] for lm in landmarks])
            sequences.append(sequence)
    return np.array(sequences)

def format_hand_data(augmented_sequences, original_data):
    formatted_data = []
    idx = 0
    for sample in original_data:
        formatted_sample = {'label': sample['label'], 'sequences': []}
        for seq in sample['sequences']:
            formatted_seq = {'seq': seq['seq'], 'landmarks': [], 'hand_label': seq['hand_label'], 'label': seq['label']}
            for i, lm in enumerate(seq['landmarks']):
                formatted_seq['landmarks'].append({
                    'id': lm['id'],
                    'x': augmented_sequences[idx, i, 0],
                    'y': augmented_sequences[idx, i, 1],
                    'z': augmented_sequences[idx, i, 2]
                })
            formatted_sample['sequences'].append(formatted_seq)
            idx += 1
        formatted_data.append(formatted_sample)
    return formatted_data

def add_noise(data, noise_level=0.01):
    return data + noise_level * np.random.randn(*data.shape)

def scale_data(data, scale_factor=0.1):
    scaling = 1 + scale_factor * (2 * np.random.rand() - 1)
    return data * scaling

def rotate_data(data, angle_range=10):
    angle = np.deg2rad(random.uniform(-angle_range, angle_range))
    rotation_matrix = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    rotated_data = np.dot(data[:, :2], rotation_matrix)
    return np.hstack((rotated_data, data[:, 2:]))

def translate_data(data, translation_range=0.1):
    translation = translation_range * (2 * np.random.rand(3) - 1)
    return data + translation

def time_warping(data, time_factor=0.1):
    original_timesteps = np.arange(data.shape[0])
    new_timesteps = np.linspace(0, data.shape[0] - 1, num=int(data.shape[0] * (1 + time_factor * (2 * np.random.rand() - 1))))
    interpolator = interp1d(original_timesteps, data, axis=0, kind='linear', fill_value="extrapolate")
    return interpolator(new_timesteps)

def augment_data(data, num_augmentations=10):
    augmented_data = []
    for _ in range(num_augmentations):
        for sequence in data:
            aug_sequence = sequence.copy()
            aug_sequence = add_noise(aug_sequence)
            aug_sequence = scale_data(aug_sequence)
            aug_sequence = rotate_data(aug_sequence)
            aug_sequence = translate_data(aug_sequence)
            aug_sequence = time_warping(aug_sequence)
            augmented_data.append(aug_sequence)
    return np.array(augmented_data)

parsed_data = parse_hand_data(hand_data)

num_augmentations = 10  
augmented_data = augment_data(parsed_data, num_augmentations=num_augmentations)
final_dataset = np.concatenate((parsed_data, augmented_data), axis=0)
final_dataset = final_dataset[:500]
final_formatted_data = format_hand_data(final_dataset, hand_data)
print(f'Final dataset size: {len(final_formatted_data)}')
print(final_formatted_data)



ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (27080,) + inhomogeneous part.

In [1]:
import numpy as np
import random
from scipy.interpolate import interp1d


def add_noise(data, noise_level=0.01):
    return data + noise_level * np.random.randn(*data.shape)

def scale_data(data, scale_factor=0.1):
    scaling = 1 + scale_factor * (2 * np.random.rand() - 1)
    return data * scaling

def rotate_data(data, angle_range=10):
    angle = np.deg2rad(random.uniform(-angle_range, angle_range))
    rotation_matrix = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    rotated_data = np.dot(data[:, :2], rotation_matrix)
    return np.hstack((rotated_data, data[:, 2:]))

def translate_data(data, translation_range=0.1):
    translation = translation_range * (2 * np.random.rand(3) - 1)
    return data + translation

def time_warping(data, time_factor=0.1):
    original_timesteps = np.arange(data.shape[0])
    new_timesteps = np.linspace(0, data.shape[0] - 1, num=int(data.shape[0] * (1 + time_factor * (2 * np.random.rand() - 1))))
    interpolator = interp1d(original_timesteps, data, axis=0, kind='linear', fill_value="extrapolate")
    return interpolator(new_timesteps)

def augment_data(data, num_augmentations=10):
    augmented_data = []
    for _ in range(num_augmentations):
        for sequence in data:
            aug_sequence = sequence.copy()
            aug_sequence = add_noise(aug_sequence)
            aug_sequence = scale_data(aug_sequence)
            aug_sequence = rotate_data(aug_sequence)
            aug_sequence = translate_data(aug_sequence)
            aug_sequence = time_warping(aug_sequence)
            augmented_data.append(aug_sequence)
    return np.array(augmented_data)

num_augmentations = 10  
augmented_data = augment_data(hand_data, num_augmentations=num_augmentations)


final_dataset = np.concatenate((hand_data, augmented_data), axis=0)

final_dataset = final_dataset[:500]

print(f'Final dataset size: {len(final_dataset)}')


NameError: name 'hand_data' is not defined

In [ ]:
def normalize_data_sequence(data, sequence_length):
    normalized_data = []
    for sequence in data:
        if len(sequence) < sequence_length:
            sequence = np.pad(sequence, ((0, sequence_length - len(sequence)), (0, 0)), mode='constant')
        elif len(sequence) > sequence_length:
            sequence = sequence[:sequence_length]
        normalized_data.append(sequence)
    return np.array(normalized_data)

In [32]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

X = np.array([seq['landmarks'] for sample in final_formatted_data for seq in sample['sequences']])
y = np.array([sample['label'] for sample in final_formatted_data for _ in sample['sequences']])


num_samples = X.shape[0]
timesteps = X.shape[1]
num_landmarks = X.shape[2]

X = X.reshape(num_samples, timesteps, num_landmarks * 3)

model = Sequential()
model.add(LSTM(units=50, input_shape=(timesteps, num_landmarks * 3)))
model.add(Dense(units=1, activation='sigmoid'))  # Adjust units and activation based on your classification needs

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(X, y, epochs=10, batch_size=32, validation_split=0.2)

loss, accuracy = model.evaluate(X, y)
print(f'Final training loss: {loss:.4f}, accuracy: {accuracy:.4f}')


NameError: name 'final_formatted_data' is not defined